In [1]:
import os
import glob
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from torchvision import transforms
import timm
from facenet_pytorch import MTCNN
from tqdm.notebook import tqdm

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"Menggunakan device: {device}")

Menggunakan device: cuda:0


### PERSIAPAN TRANSFORMASI

In [2]:
class_names = ['fake_mannequin', 'fake_mask', 'fake_printed', 'fake_screen', 'fake_unknown', 'realperson']

transform_300 = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

transform_224 = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

### LOAD GOLDEN MODELS (B3 & ConvNeXt-Tiny Asli)

In [3]:
print("Memuat EfficientNet-B3 (Bobot 35%)...")
model_eff = timm.create_model('efficientnet_b3', pretrained=False, num_classes=len(class_names))
model_eff.load_state_dict(torch.load('../models/efficientnet_b3_model.pth', map_location=device))
model_eff = model_eff.to(device).eval()

print("Memuat ConvNeXt-Tiny (Bobot 65%)...")
model_conv = timm.create_model('convnext_tiny', pretrained=False, num_classes=len(class_names))
model_conv.load_state_dict(torch.load('../models/convnext_tiny_model.pth', map_location=device))
model_conv = model_conv.to(device).eval()

Memuat EfficientNet-B3 (Bobot 35%)...


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_31012\795431906.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_eff.load_state_dict(torch.load('../models/efficientnet

Memuat ConvNeXt-Tiny (Bobot 65%)...


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_31012\795431906.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_conv.load_state_dict(torch.load('../models/convnext_ti

### MULTI-ZOOM MTCNN

In [4]:
mtcnn_tight = MTCNN(keep_all=False, select_largest=True, margin=20, post_process=False, device=device)
mtcnn_normal = MTCNN(keep_all=False, select_largest=True, margin=60, post_process=False, device=device)
mtcnn_wide = MTCNN(keep_all=False, select_largest=True, margin=100, post_process=False, device=device)

class_weights = torch.tensor([1.15, 1.15, 1.15, 1.15, 1.15, 0.90]).to(device)
print("Sistem Anti-Bias Panitia Aktif!")

d:\Andri\findIT-AntiSpoofing\.venv\Lib\site-packages\facenet_pytorch\models\mtcnn.py:34: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(state_dict_pat

Sistem Anti-Bias Panitia Aktif!


### PROSES INFERENCE

In [5]:
test_dir = '../Data/test/' 
submission_df = pd.read_csv('../Data/samplesubmission.csv')
predictions = []

print("Memulai Prediksi Golden Ensemble + Threshold Shifting...")
with torch.no_grad():
    for img_id in tqdm(submission_df['id']):
        search_pattern = os.path.join(test_dir, f"{img_id}.*")
        matching_files = glob.glob(search_pattern)
        
        if not matching_files:
            predictions.append('fake_unknown')
            continue
            
        img_path = matching_files[0] 
        
        try:
            img = Image.open(img_path).convert('RGB')
            faces = [mtcnn_tight(img), mtcnn_normal(img), mtcnn_wide(img)]
            
            all_probs_eff, all_probs_conv = [], []
            
            for i, face in enumerate(faces):
                if face is not None:
                    pil_img = transforms.ToPILImage()(face.byte())
                else:
                    # Smart Center Crop
                    w, h = img.size
                    ratio = 0.4 if i == 0 else (0.6 if i == 1 else 0.8)
                    margin = (1.0 - ratio) / 2.0
                    left, top = w * margin, h * margin
                    right, bottom = w * (1.0 - margin), h * (1.0 - margin)
                    pil_img = img.crop((left, top, right, bottom))
                    
                t_300 = transform_300(pil_img).unsqueeze(0).to(device)
                t_224 = transform_224(pil_img).unsqueeze(0).to(device)
                
                all_probs_eff.append(F.softmax(model_eff(t_300), dim=1))
                all_probs_conv.append(F.softmax(model_conv(t_224), dim=1))
            
            # Rata-rata dari 3 level zoom
            avg_eff = torch.mean(torch.stack(all_probs_eff), dim=0)
            avg_conv = torch.mean(torch.stack(all_probs_conv), dim=0)
            
            # Weighted Voting Model (ConvNeXt 65%, B3 35%)
            base_prob = (avg_conv * 0.65) + (avg_eff * 0.35)
            
            # --- PENERAPAN THRESHOLD SHIFTING ---
            # Kalikan probabilitas asli dengan bobot manipulasi kita
            shifted_prob = base_prob * class_weights
            
            # Ambil kelas dengan nilai tertinggi (setelah dimanipulasi)
            _, predicted_idx = torch.max(shifted_prob, 1)
            predicted_class = class_names[predicted_idx.item()]
            
            predictions.append(predicted_class)
            
        except Exception as e:
            print(f"Error memproses {img_path}: {e}")
            predictions.append('fake_unknown')

Memulai Prediksi Golden Ensemble + Threshold Shifting...


  0%|          | 0/404 [00:00<?, ?it/s]

d:\Andri\findIT-AntiSpoofing\.venv\Lib\site-packages\PIL\Image.py:1034: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


### SIMPAN CSV

In [6]:
submission_dir = '../Data/submission/'
os.makedirs(submission_dir, exist_ok=True)
submission_df['label'] = predictions

file_name = 'submission_findit_golden_shifted.csv'
submission_file_path = os.path.join(submission_dir, file_name)
submission_df.to_csv(submission_file_path, index=False)

print("="*40)
print(f"File submission Threshold Shifted berhasil disimpan!")
print(f"Lokasi: {submission_file_path}")
print("="*40)

File submission Threshold Shifted berhasil disimpan!
Lokasi: ../Data/submission/submission_findit_golden_shifted.csv
